# Neutrino Flavor Isospin and Open Quantum Systems

Neutrinos are produced and detected as flavor eigenstates, but propagate as mass eigenstates with distinct dispersion relations. Under plane-wave propagation, the density operator is an element of the self-reverse subspace `State: 1 x y z`, with isospin vector `P = 2 * rho.select[1]`. Spatial evolution is generated by Liouvillian superoperators `Rates: State <- State`, and finite propagation over distance is an extensor transfer map `Evolution: State <- State`. Wavepacket separation across astrophysical distances acts as Lindblad dephasing, damping coherent flavor oscillations into an incoherent mixture of mass eigenstates.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git"], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator

import numpy as np
from numga import NumpyContext
from numga.algebras import VGA3D
from examples.quantum.neutrino_isospin import render

np.set_printoptions(precision=3, suppress=True)

# The geometric algebra of three-dimensional Euclidean space.
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()

# Density operators and superoperators:
State = ga.gatype.self_reverse()                                             # 1 x y z
Rates = ga.gatype((State, State))                                            # State <- State
Evolution = ga.gatype((State, State))                                        # State <- State

one = mv.scalar([1.0])
flavor_e = mv.z                                                              # [] Vector (+z: electron neutrino)
flavor_mu = -mv.z                                                            # [] Vector (-z: muon neutrino)

# Coordinate bivector planes:
phase_plane = mv.x ^ mv.y                                                    # [] Bivector (quantum phase accumulation)
transition_plane = mv.y ^ mv.z                                               # [] Bivector (flavor conversion)
mixing_plane = mv.x ^ mv.z                                                   # [] Bivector (mass-flavor mixing)

## 1. Coherent Oscillations as Extensor Transfer Maps

A neutrino state is a density operator `state(P) = 0.5 * (1 + P)`. The Hamiltonian generator turns the state by its commutator with the Hamiltonian bivector. The transfer map over distance `dx` is the extensor exponential `evolution(rates, dx)`, an operator mapping `State <- State`.

In [ ]:
def state(bloch: Vector) -> State:
    """Construct density operator state: 0.5 * (1 + P)."""
    return 0.5 * (one + bloch)


def isospin(rho: State) -> Vector:
    """Extract isospin vector P from density operator: 2 * rho.select[1]."""
    return 2.0 * rho.select[1]


def evolution(rates: Rates, dx: float) -> Evolution:
    """Quantum transfer map over distance dx: exp(rates * dx) as a composed extensor map."""
    small = rates * dx                                                     # State <- State
    term = total = State
    for order in range(1, 6):
        term = small(term) / order                                         # State <- State
        total = total + term
    return total                                                           # State <- State


def propagate(initial_state: State, transfer: Evolution, steps: int) -> Iterator[Vector]:
    """Yield isospin vector of state propagating under constant transfer map."""
    rho = initial_state
    for _ in range(steps):
        yield isospin(rho)                                                  # [] Vector
        rho = transfer(rho)                                                 # State <- State


theta_deg = 30.0
theta = np.radians(theta_deg)
omega = 1.0

# Vacuum Hamiltonian bivector: phase plane tilted by mixing angle theta:
mixing_rotor = (mixing_plane * (-theta)).exp()                               # [] Rotor
b_vac = omega * (mixing_rotor >> (-phase_plane))                            # [] Bivector

steps = 300
wavelength = 4.0 * np.pi / omega
distances, dx = np.linspace(0.0, 2.0 * wavelength, steps, retstep=True)

# Generator is an extensor superoperator: - b_vac.commutator(State)
turning = - b_vac.commutator(State).cast(Rates)                             # State <- State
transfer_vac = evolution(turning, dx)                                       # State <- State

render.draw_precession(
    f"Vacuum Oscillations (theta={int(theta_deg)} deg)",
    propagate(state(flavor_e), transfer_vac, steps),
    b_vac,
    distances,
);

## 2. Wavepacket Separation and Lindblad Decoherence

Because mass eigenstates have different group velocities, wavepackets separate over astrophysical propagation distances. This spatial separation acts as pure dephasing along the mass axis, damping off-diagonal mass coherence while strictly conserving probability trace. The state vector spirals into the interior of the flavor sphere, settling into an incoherent mixture of mass eigenstates with constant asymptotic flavor probabilities.

In [ ]:
def relaxation(process: Vector) -> Rates:
    """Lindblad dissipator: (l >> State) - (l * l).anticommutator(State)."""
    back = process.reverse().symmetric_reverse_product()                  # [] State
    return (process >> State) - back.anticommutator(State)                 # State <- State


# Mass eigenstate direction:
mass_axis = mixing_rotor >> flavor_mu                                       # [] Vector

gamma = 0.06                                                               # dephasing rate per unit length
dephasing = gamma * relaxation(mass_axis)                                   # State <- State
generator_open = turning + dephasing                                        # State <- State

steps_dec = 400
distances_dec, dx_dec = np.linspace(0.0, 100.0, steps_dec, retstep=True)
transfer_dec = evolution(generator_open, dx_dec)                            # State <- State

asymptotic_pe = float(1.0 - 0.5 * np.sin(2.0 * theta) ** 2)

render.draw_decoherence(
    "Wavepacket Decoherence",
    propagate(state(flavor_e), transfer_dec, steps_dec),
    b_vac,
    distances_dec,
    asymptotic_pe,
);

## 3. Solar MSW Adiabatic Conversion

In the Sun, electron density falls off exponentially with radius. As the Hamiltonian plane tilts adiabatically from the phase plane toward the vacuum plane, the state vector is trapped and guided into a muon neutrino.

In [ ]:
def adiabatic_transfer(
    initial_state: State,
    rates_profile: Iterable[tuple[Rates, Bivector]],
    dx: float,
) -> Iterator[tuple[Vector, Bivector]]:
    """Yield isospin vector and Hamiltonian plane under varying generator profile."""
    rho = initial_state
    for rates, plane in rates_profile:
        yield isospin(rho), plane                                           # [] Vector, Bivector
        step_map = evolution(rates, dx)                                     # State <- State
        rho = step_map(rho)                                                 # State <- State


theta_small = np.radians(10.0)
mixing_small = (mixing_plane * (-theta_small)).exp()
b_vac_small = omega * (mixing_small >> (-phase_plane))

v_res = float(omega * np.cos(2.0 * theta_small))
steps_sun = 600
distances_sun, dx_sun = np.linspace(0.0, 100.0, steps_sun, retstep=True)
v_profile = 3.0 * v_res * np.exp(-distances_sun / 25.0)

# Local generator and plane at each step along the solar profile:
planes_sun = [b_vac_small + phase_plane * v for v in v_profile]
rates_sun = [(- b.commutator(State).cast(Rates), b) for b in planes_sun]

render.draw_adiabatic(
    "Solar MSW (Adiabatic Conversion)",
    adiabatic_transfer(state(flavor_e), rates_sun, dx_sun),
    distances_sun,
    v_profile,
);

In [ ]:
# checks
from numga import stack

# 1. Vacuum transfer map match against textbook formula:
traj_vac = stack(list(propagate(state(flavor_e), transfer_vac, steps)), axis=0)
pe_vac = 0.5 * (1.0 + (traj_vac | flavor_e).to_array())
analytical_vac = 1.0 - np.sin(2.0 * theta) ** 2 * np.sin(0.5 * omega * distances) ** 2
np.testing.assert_allclose(pe_vac, analytical_vac, atol=1e-5)

# 2. Wavepacket decoherence asymptotic match:
traj_dec = stack(list(propagate(state(flavor_e), transfer_dec, steps_dec)), axis=0)
pe_dec = 0.5 * (1.0 + (traj_dec | flavor_e).to_array())
np.testing.assert_allclose(pe_dec[-1], asymptotic_pe, atol=1e-3)

# 3. Solar MSW adiabatic conversion:
sun_history = list(adiabatic_transfer(state(flavor_e), rates_sun, dx_sun))
traj_sun = stack([s for s, _ in sun_history], axis=0)
pe_sun = 0.5 * (1.0 + (traj_sun | flavor_e).to_array())
assert pe_sun[0] > 0.99
assert pe_sun[-1] < 0.05

# 4. State scalar trace conservation:
s_test = state(flavor_e)
for _ in range(50):
    s_test = transfer_dec(s_test)
    assert np.abs(s_test.select[0].to_array() - 0.5) < 1e-12

print("All checks passed successfully!")